هذه نسخة يومية مستخرجة من دفتر سلطان المجمع المرفق. حُفظ الكود والمخرجات وأرقام التنفيذ كما وردت، ولم تُعد الخلايا للتشغيل أثناء التقسيم. الأصل الكامل محفوظ في [دفتر المشروع المجمع](../notebooks/Sultan_Training_Project.ipynb).

بيانات `metadata.masar` وتوقيتات `metadata.execution` القديمة موروثة من دفتر الدورة المرجعي؛ أزيلت من المقتطف حتى لا تُنسب إلى تشغيل سلطان. تبقى في الأصل غير المعدل لأغراض التتبع. راجع [سجل التقسيم](../notebooks/README.md).

<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Day 3 · Delta transactions and maintenance</h1><p>Apply a correction once; preserve revision precedence; read earlier versions; test schema changes and maintenance on isolated copies.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اليوم 3 · معاملات Delta والصيانة</h1><p>طبّق التصحيح مرة واحدة، واحفظ أولوية المراجعات، واقرأ النسخ السابقة، واختبر تغيير المخطط والصيانة على نسخ معزولة.</p></td></tr></table>



<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Continue your project</h2><p>Use the same repository and successful Day 1 workspace. Restore your handoff ZIP at the repository root when using a new session. Read <a href="README.md">today’s guide</a> before running all cells in order.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>١. استكمل مشروعك</h2><p>استخدم المستودع نفسه ومساحة اليوم الأول الناجحة. استعد ملف الانتقال في جذر المستودع عند استخدام جلسة جديدة. اقرأ <a href="README.md">دليل اليوم</a> ثم شغّل الخلايا بالترتيب.</p></td></tr></table>



In [26]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))


Continue workspace: outputs/day01_bronze_jmbv27q9


<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>04a · Correct and read versions</h2><p>Follow <a href="labs/lab04/WALKTHROUGH.md">the lab walkthrough</a>. The cell runs real operations and saves reports; inspect the checks and explain one observation in your lab notes.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>04a · صحّح واقرأ النسخ</h2><p>اتبع <a href="labs/lab04/WALKTHROUGH.md">شرح اللاب</a>. تنفذ الخلية العمليات وتحفظ تقاريرها؛ افحص النتائج وفسّر ملاحظة واحدة في ملف اللاب.</p></td></tr></table>



In [27]:
from masar.delta_lab import run_transactions_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_transactions_lab(spark, SOURCE, WORK)
    validate_stage_result('lab04a_transactions', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print(json.dumps({k: result[k] for k in ('before','after_correction','past_version_read')}, indent=2, default=str))
finally:
    spark.stop()


{
  "scope": "DAY03_TRANSACTIONS_ENGINE",
  "checks": {
    "native_correction_matches_source_expectation": true,
    "business_rows_stay_75": true,
    "replay_and_stale_delivery_preserve_values": true,
    "same_revision_conflict_rejected": true,
    "actual_prior_version_read": true,
    "mixed_valid_invalid_batch_rejected_atomically": true
  }
}
{
  "before": {
    "rows": 75,
    "business_digest": "0d16e2795620ae0c0f54d3fcd52a5b13fb2c2a47cd4d0c42c8ddb195f19bc6e0",
    "version": 1,
    "schema": [
      [
        "trip_id",
        "string"
      ],
      [
        "driver_id",
        "string"
      ],
      [
        "city",
        "string"
      ],
      [
        "start_utc",
        "timestamp"
      ],
      [
        "end_utc",
        "timestamp"
      ],
      [
        "trip_date_local",
        "date"
      ],
      [
        "fare_sar",
        "decimal(12,2)"
      ],
      [
        "distance_km",
        "decimal(12,2)"
      ],
      [
        "duration_seconds",

<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>04b · Test safe maintenance</h2><p>Follow <a href="labs/lab04/WALKTHROUGH.md">the lab walkthrough</a>. The cell runs real operations and saves reports; inspect the checks and explain one observation in your lab notes.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>04b · اختبر الصيانة الآمنة</h2><p>اتبع <a href="labs/lab04/WALKTHROUGH.md">شرح اللاب</a>. تنفذ الخلية العمليات وتحفظ تقاريرها؛ افحص النتائج وفسّر ملاحظة واحدة في ملف اللاب.</p></td></tr></table>



In [28]:
from masar.delta_lab import run_maintenance_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_maintenance_lab(spark, SOURCE, WORK)
    validate_stage_result('lab04b_maintenance', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print(json.dumps({'recovery': result['recovery'], 'vacuum': result['vacuum']}, indent=2, default=str))
finally:
    spark.stop()


{
  "scope": "DAY03_MAINTENANCE_ENGINE",
  "checks": {
    "unexpected_column_rejected": true,
    "approved_evolution_preserves_business_values": true,
    "compaction_preserves_values": true,
    "delete_affects_copy_only": true,
    "restore_creates_new_commit": true,
    "vacuum_is_non_destructive_dry_run": true,
    "trusted_silver_unchanged": true
  }
}
{
  "recovery": {
    "before": {
      "rows": 75,
      "business_digest": "1321d375742d806a1f0fef82be9e2862af04f5d18f3a976792a6b1d9aa1b4383",
      "version": 0,
      "schema": [
        [
          "trip_id",
          "string"
        ],
        [
          "driver_id",
          "string"
        ],
        [
          "city",
          "string"
        ],
        [
          "start_utc",
          "timestamp"
        ],
        [
          "end_utc",
          "timestamp"
        ],
        [
          "trip_date_local",
          "date"
        ],
        [
          "fare_sar",
          "decimal(12,2)"
        ],
       

<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Review and save</h2><p>Answer <a href="PRACTICE.md">the questions</a> as part of the existing lab notes, then use <a href="COMPLETION.md">the completion checklist</a>. Save your notebook with actual outputs. The archive below is a handoff, not another assignment.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>راجع واحفظ</h2><p>أجب عن <a href="PRACTICE.md">الأسئلة</a> ضمن ملاحظات اللاب، ثم استخدم <a href="COMPLETION.md">قائمة الاكتمال</a>. احفظ دفترك بالمخرجات الفعلية. ملف الانتقال أدناه ليس تكليفًا آخر.</p></td></tr></table>



In [29]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day03_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))


Retain the notebook outputs, notes and outputs/day03_handoff.zip
